# GraphX và GraphFrames trên dữ liệu preprocessing chung

Input là output của upstream IBM AML HI-Small, không đọc hoặc làm sạch CSV gốc.
Chạy bằng kernel Scala 2.12.18 - GraphX benchmark, rồi mở notebook 02 để xem kết quả.
Môi trường: Spark 3.5.7, Scala 2.12.18, Java 17, GraphFrames 0.10.0.
Restart kernel trước mỗi lần Run All.

## Cách đọc code

`val` khai báo giá trị không gán lại; `var` dùng cho state thay đổi sau mỗi vòng.
`for` và `while` chạy tuần tự trên driver.
Các phép `select`, `join`, `map` và `aggregateMessages` là API Spark xử lý dữ liệu phân tán.
Tuple `(sender, receiver, amount)` chứa ba giá trị.
`startPhase` bắt đầu timer; `finishPhase` ghi thời gian sau khi Spark thực sự tính xong.
`persist` giữ dữ liệu để dùng lại; `checkpoint` cắt chuỗi phụ thuộc; `unpersist` giải phóng cache.
`col("id")` và `$"id"` cùng tham chiếu tới cột `id`; `lit(0.0)` tạo giá trị hằng trong biểu thức Spark.
`getOrElse(0.0)` dùng 0 khi một vertex không có message nhận được.


## Cấu hình

Mặc định dùng toàn bộ `vertices.parquet` và `edges.parquet`, không lấy mẫu hoặc gom cạnh.
Đặt biến môi trường `AML_PROCESSED_DIR` để dùng thư mục upstream ngoài repo.
Mỗi framework có một warmup và ba lần đo, cùng số partition và số vòng PageRank.

In [1]:
import java.io.{File, PrintWriter}
import java.nio.file.Files
import java.lang.management.ManagementFactory
import java.security.MessageDigest
import scala.collection.JavaConverters._

// Tìm thư mục dự ávertexCount từ root hoặc notebooks/.
var projectDirectory = new File(".").getCanonicalFile
while (projectDirectory != null && !new File(projectDirectory, "upstream.json").isFile) {
  projectDirectory = projectDirectory.getParentFile
}
require(projectDirectory != null, "Cannot find upstream.json")
val root = projectDirectory
val partitionCount = 4
val iterations = 5
val repeats = 3
val warmups = 1
val seed = 42L
val damping = 0.85
val master = "local[2]"
val defaultDataDirectory = new File(root, "data/processed").getPath
val dataDirectory = sys.env.getOrElse("AML_PROCESSED_DIR", defaultDataDirectory)
val sourceDir = new File(dataDirectory).getCanonicalFile
val reportRoot = new File(root, "reports")
val output = new File(reportRoot, "runs/scala_" + System.currentTimeMillis())
val eventDir = new File(output, "events")
val inputDir = new File(root, "data/benchmark/" + output.getName)
eventDir.mkdirs()
inputDir.mkdirs()
require(partitionCount >= 2 && iterations > 0 && repeats >= 2 && warmups >= 1)
println(s"Run: ${output.getAbsolutePath}")

Run: /home/tuanta/BigData/graphX/reports/runs/scala_1792579688023


### Nạp Spark và GraphFrames

Các JAR được nạp từ môi trường đã cài trong dự án để kernel và Spark dùng cùng dependency.
`almond-spark` hỗ trợ chạy Spark tương tác và gửi class tạo trong notebook tới executor.
Phần này chỉ là thiết lập môi trường, không nằm trong benchmark.

In [2]:
interp.load.cp {
  val sparkHome = os.Path(sys.env("SPARK_HOME"))
  os.list(sparkHome / "jars").filter(_.ext == "jar").filterNot { jar =>
    Seq("scala-library-", "scala-reflect-", "scala-compiler-", "spark-repl_")
      .exists(jar.last.startsWith)
  }
}

In [3]:
import $ivy.`sh.almond::almond-spark:0.14.0-RC8`
interp.load.cp(Seq(
  os.Path(new File(root, ".tools/jars/graphframes-spark3_2.12-0.10.0.jar")),
  os.Path(new File(root, ".tools/jars/graphframes-graphx-spark3_2.12-0.10.0.jar"))
))

In [4]:
import org.apache.spark.sql._
import org.apache.spark.sql.functions._
import org.apache.spark.graphx._
import org.apache.spark.rdd.RDD
import org.apache.spark.storage.StorageLevel
import org.graphframes.GraphFrame
import org.graphframes.lib.AggregateMessages

val spark = NotebookSparkSession.builder()
  .master(master).appName("GraphX-vs-GraphFrames-native-Scala")
  .config("spark.eventLog.enabled", "true")
  .config("spark.eventLog.compress", "false")
  .config("spark.eventLog.dir", eventDir.toURI.toString)
  .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
  .config("spark.sql.shuffle.partitions", partitionCount)
  .config("spark.sql.adaptive.enabled", "false")
  .config("spark.sql.autoBroadcastJoinThreshold", -1)
  .config("spark.sql.adaptive.autoBroadcastJoinThreshold", -1)
  .getOrCreate()
val sc = spark.sparkContext
import spark.implicits._
val storageLevel = StorageLevel.MEMORY_AND_DISK
sc.setCheckpointDir(new File(output, "checkpoints").getAbsolutePath)
sc.setLogLevel("ERROR")
require(spark.version == "3.5.7")
require(scala.util.Properties.versionNumberString == "2.12.18")
println(s"Spark ${spark.version}; Scala ${scala.util.Properties.versionNumberString}; UI ${sc.uiWebUrl}")

Loading spark-stubs

Failed to download https://repo.maven.apache.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.1/hadoop-client-runtime-3.3.1-sources.jar


Failed to download https://repo.maven.apache.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.1/hadoop-client-runtime-3.3.1-sources.jar.sha1


Creating SparkSession

Spark UI

Spark 3.5.7; Scala 2.12.18; UI Some(http://localhost:4040)


## Input upstream

Giữ nguyên USD amount, định danh và tập giao dịch do upstream quyết định.
Không dùng `edges_agg.parquet` hoặc các thuộc tính tổng tiền đã tính sẵn trên vertices.
Kiểm tra schema, số dòng, self-loop và endpoint trước khi chạy.
Input không hợp lệ làm notebook dừng; không tự sửa dữ liệu.
Đọc Parquet, kiểm tra và ánh xạ ID nằm ngoài timer.

In [5]:
val mapper = new com.fasterxml.jackson.databind.ObjectMapper()
val upstream = mapper.readTree(new File(root, "upstream.json"))
val summaryFile = new File(sourceDir, "preprocess_summary.json")
require(summaryFile.isFile, s"Missing $summaryFile. See README.md for upstream preprocessing.")
val upstreamSummary = mapper.readTree(summaryFile)
val rawVertices = spark.read.parquet(new File(sourceDir, "vertices.parquet").getPath)
val rawEdges = spark.read.parquet(new File(sourceDir, "edges.parquet").getPath)
require(rawVertices.schema("id").dataType == org.apache.spark.sql.types.StringType)
for (column <- Seq("src", "dst")) {
  require(rawEdges.schema(column).dataType == org.apache.spark.sql.types.StringType)
}
require(rawEdges.schema("amount").dataType == org.apache.spark.sql.types.DoubleType)
val timestampType = rawEdges.schema("timestamp").dataType.typeName
require(timestampType == "timestamp" || timestampType == "timestamp_ntz", "Invalid timestamp type")
val sourceVertices = rawVertices.select($"id".as("source_id")).persist(storageLevel)
val selected = rawEdges.select("src", "dst", "amount", "timestamp").persist(storageLevel)
val sourceVertexCount = sourceVertices.count()
val selectedCount = selected.count()
require(sourceVertexCount > 0 && selectedCount > 0)
for (key <- Seq("vertices", "edges")) {
  require(upstreamSummary.has(key), "Summary missing " + key)
}
require(sourceVertexCount == upstreamSummary.get("vertices").asLong())
require(selectedCount == upstreamSummary.get("edges").asLong())
require(sourceVertices.filter($"source_id".isNull).count() == 0)
require(sourceVertices.distinct().count() == sourceVertexCount, "Duplicate vertex IDs")
require(selected.filter($"src".isNull || $"dst".isNull || $"timestamp".isNull ||
  $"amount".isNull || isnan($"amount") || abs($"amount") === Double.PositiveInfinity).count() == 0)
require(selected.filter($"src" === $"dst").count() == 0, "Upstream edges must exclude self-loops")
println(s"Upstream input: $sourceVertexCount vertices, $selectedCount transaction edges")

parquet at cmd5.sc:6

parquet at cmd5.sc:7

count at cmd5.sc:17

count at cmd5.sc:17

count at cmd5.sc:18

...

count at cmd5.sc:28

count at cmd5.sc:28

count at cmd5.sc:29

count at cmd5.sc:29

Upstream input: 422734 vertices, 4487124 transaction edges


### Ánh xạ ID cho GraphX

Sắp xếp ID upstream và đánh số Long từ 1, giữ `source_id` để đối chiếu.
Cả hai framework dùng cùng bảng ánh xạ, cùng vertices và cùng transaction edges.
Cạnh song song được giữ nguyên; tài khoản không bị gộp thêm.
Hai bảng chung được lưu cùng mỗi run để tái lập input chính xác.

In [6]:
val vertexData = sourceVertices.orderBy("source_id").as[String].rdd.zipWithIndex()
  .map { case (sourceId, index) => (index + 1L, sourceId) }.toDF("id", "source_id")
val senderIds = vertexData.select($"id".as("src"), $"source_id".as("source_src"))
val receiverIds = vertexData.select($"id".as("dst"), $"source_id".as("source_dst"))
val edgeData = selected.withColumnRenamed("src", "source_src").withColumnRenamed("dst", "source_dst")
  .join(senderIds, Seq("source_src")).join(receiverIds, Seq("source_dst"))
  .select("src", "dst", "amount", "timestamp")
vertexData.repartition(partitionCount).write.parquet(new File(inputDir, "vertices").getAbsolutePath)
edgeData.repartition(partitionCount).write.parquet(new File(inputDir, "edges").getAbsolutePath)
selected.unpersist(blocking = true)
sourceVertices.unpersist(blocking = true)
val vertices = spark.read.parquet(new File(inputDir, "vertices").getAbsolutePath).persist(storageLevel)
val edges = spark.read.parquet(new File(inputDir, "edges").getAbsolutePath).persist(storageLevel)
val vertexCount = vertices.count()
val edgeCount = edges.count()
require(vertexCount == sourceVertexCount && edgeCount == selectedCount, "Missing edge endpoints")
require(vertices.select("id").distinct().count() == vertexCount)
val baseCacheIds = sc.getPersistentRDDs.keySet.toSet
println(s"Common input: $vertexCount vertices, $edgeCount edges")

rdd at cmd6.sc:1

rdd at cmd6.sc:1

zipWithIndex at cmd6.sc:1

parquet at cmd6.sc:8

parquet at cmd6.sc:8

...

count at cmd6.sc:15

count at cmd6.sc:17

count at cmd6.sc:17

count at cmd6.sc:17

Common input: 422734 vertices, 4487124 edges


## 3. Ranh giới phép đo

`build` gồm tạo representation, repartition và materialize cache.
`aggregate` gồm gửi message, cộng và materialize kết quả.
`pr_setup` gồm degree, trọng số cạnh và checkpoint chuẩn bị; mỗi `pr_iteration` gồm dangling reduction, message, update và checkpoint state.
Các chức năng đo thời gian dưới đây chỉ bọc bên ngoài thuật toán, không thay thế GraphX API.
Cache được đo bằng Spark cached blocks, không phải tổng RSS hoặc peak heap.

In [7]:
val metrics = new PrintWriter(new File(output, "phase_timings.csv"))
val checks = new PrintWriter(new File(output, "correctness.csv"))
val loads = new PrintWriter(new File(output, "partition_load.csv"))
val progress = new PrintWriter(new File(output, "progress.log"))
var currentRun = ""
var expectedIncoming = Map.empty[Long, Double]
var expectedRanks = Map.empty[Long, Double]

expectedRanks: Map[Long, Double] = Map()

In [8]:
def csv(writer: PrintWriter, fields: Seq[Any]): Unit = {
  val escapedFields = scala.collection.mutable.ArrayBuffer.empty[String]
  for (field <- fields) {
    val escapedText = field.toString.replace("\"", "\"\"")
    escapedFields += "\"" + escapedText + "\""
  }
  writer.println(escapedFields.mkString(","))
  writer.flush()
}

// Các phase chạy tuầvertexCount tự, chỉ có một timer hoạt động tại một thời điểm.
var phaseName = ""
var phaseIteration = 0
var phaseStarted = 0L
var phaseGcStarted = 0L
var phaseMemoryStarted = 0L
var phaseDiskStarted = 0L

def gcMillis: Long = {
  var total = 0L
  for (collector <- ManagementFactory.getGarbageCollectorMXBeans.asScala) {
    val elapsed = collector.getCollectionTime
    if (elapsed >= 0) {
      total += elapsed
    }
  }
  total
}

def cacheBytes: Long = {
  var total = 0L
  for (storage <- sc.getRDDStorageInfo) {
    total += storage.memSize
  }
  total
}

def diskBytes: Long = {
  var total = 0L
  for (storage <- sc.getRDDStorageInfo) {
    total += storage.diskSize
  }
  total
}

def startPhase(name: String, iteration: Int = 0): Unit = {
  phaseName = name
  phaseIteration = iteration
  val jobGroup = currentRun + "|" + name + "|" + iteration
  sc.setJobGroup(jobGroup, jobGroup)
  phaseStarted = System.nanoTime()
  phaseGcStarted = gcMillis
  phaseMemoryStarted = cacheBytes
  phaseDiskStarted = diskBytes
}

def finishPhase(): Unit = {
  val elapsedMillis = (System.nanoTime() - phaseStarted) / 1e6
  val memoryNow = cacheBytes
  val diskNow = diskBytes
  val gcElapsed = gcMillis - phaseGcStarted
  csv(metrics, Seq(currentRun, phaseName, phaseIteration, elapsedMillis,
    memoryNow - phaseMemoryStarted, memoryNow, diskNow - phaseDiskStarted, gcElapsed))
  sc.clearJobGroup()
  progress.println(currentRun + " " + phaseName + " " + phaseIteration + ": " + elapsedMillis + " ms")
  progress.flush()
}

def writeText(name: String, text: String): Unit = {
  val writer = new PrintWriter(new File(output, name))
  try {
    writer.println(text)
  } finally {
    writer.close()
  }
}



phaseDiskStarted: Long = 0L

In [9]:
def validateResult(kind: String, result: Map[Long, Double], expected: Map[Long, Double]): Unit = {
  for ((id, value) <- result) {
    require(expected.contains(id), "Unexpected vertex: " + id)
    require(!value.isNaN && !value.isInfinity, "Non-finite result at vertex " + id)
  }
  if (kind == "pagerank") {
    require(result.size == expected.size, "Missing PageRank vertices")
  }
  var maxAbsoluteError = 0.0
  var maxScaledError = 0.0
  for ((id, expectedValue) <- expected) {
    // Vertex không nhậvertexCount message có tổng tiềvertexCount nhậvertexCount bằng 0.
    val actualValue = result.getOrElse(id, 0.0)
    val absoluteError = math.abs(actualValue - expectedValue)
    val scaledError = absoluteError / math.max(1.0, math.abs(expectedValue))
    maxAbsoluteError = math.max(maxAbsoluteError, absoluteError)
    maxScaledError = math.max(maxScaledError, scaledError)
  }
  val total = result.values.sum
  require(maxScaledError < 1e-9, kind + " scaled error: " + maxScaledError)
  if (kind == "pagerank") {
    require(math.abs(total - 1.0) < 1e-9, "PageRank total: " + total)
  }
  csv(checks, Seq(currentRun, kind, result.size, maxAbsoluteError, maxScaledError, total, true))
}

def cleanup(): Unit = {
  // Giữ cache input chung, chỉ xóa cache do lượt benchmark tạo ra.
  for ((id, cachedRDD) <- sc.getPersistentRDDs) {
    if (!baseCacheIds.contains(id)) {
      cachedRDD.unpersist(blocking = true)
    }
  }
}

def saveLoad(counts: Seq[(Int, Long)]): Unit = {
  val edgeCounts = counts.toMap
  for (partitionId <- 0 until partitionCount) {
    val count = edgeCounts.getOrElse(partitionId, 0L)
    csv(loads, Seq(currentRun, partitionId, count))
  }
}


In [10]:
csv(metrics, Seq("run", "phase", "iteration", "wall_ms", "cache_delta_bytes", "cache_total_bytes", "disk_delta_bytes", "jvm_gc_ms"))
csv(checks, Seq("run", "kind", "vertices_returned", "max_abs_error", "max_scaled_error", "sum", "passed"))
csv(loads, Seq("run", "partition_id", "edges"))

## 4. GraphX: bắt đầu từ Graph, VertexRDD và EdgeRDD

`Graph(vertexRDD, edgeRDD, defaultValue)` là constructor chính trong guide.
Các tham số storage level thêm vào nhằm giữ cùng cache policy với GraphFrames.
Không tạo một RDD join pipeline để giả làm GraphX; mọi phép tính bên dưới dùng graph operators chính thức.

In [11]:
def buildGraphX(vertexData: DataFrame, edgeData: DataFrame, partitions: Int): Graph[Double, Double] = {
  val vertexIds = vertexData.select("id").as[Long].rdd
  val vertexRDD = vertexIds.map(id => (id, 1.0))

  val transfers = edgeData.select("src", "dst", "amount").as[(Long, Long, Double)].rdd
  val edgeRDD = transfers.map { case (sender, receiver, amount) =>
    Edge(sender, receiver, amount)
  }

  val graph = Graph(vertexRDD, edgeRDD,
    defaultVertexAttr = 1.0,
    edgeStorageLevel = storageLevel,
    vertexStorageLevel = storageLevel)
  val partitionedGraph = graph.partitionBy(PartitionStrategy.EdgePartition2D, partitions)
  partitionedGraph.persist(storageLevel)
}


### Incoming amount: gửi một message cho mỗi cạnh

Message là số tiền trên cạnh, gửi tới `dst` bằng `sendToDst`.
Hàm merge chỉ cộng hai số Double.
`TripletFields.None` cho biết không cần lấy thuộc tính vertex.

In [12]:
def incomingAmounts(graph: Graph[Double, Double]): VertexRDD[Double] = {
    graph.aggregateMessages[Double](
      sendMsg = edge => edge.sendToDst(edge.attr),
      mergeMsg = (left, right) => left + right,
      tripletFields = TripletFields.None)
  }

### PageRank: thiết lập state bằng property operators

Với N vertex, d = 0.85, out-degree k(u) đếm cả cạnh song song và dangling mass D(t):

\[
r_v^{(t+1)} = (1-d)/N + d\left(\sum_{u\to v}r_u^{(t)}/k(u) + D^{(t)}/N\right).
\]

Rank khởi tạo bằng 1/N; tổng rank phải xấp xỉ 1 sau mỗi vòng.
Theo pattern trong guide, `outerJoinVertices` bổ sung degree, `mapTriplets` tạo trọng số cạnh và `mapVertices` khởi tạo rank.
State là `Graph[Double, Double]`, không cần tuple rank/degree trong mỗi vòng.
Danh sách dangling vertex được giữ riêng để tránh tính lại degree.

In [13]:
def preparePageRank(graph: Graph[Double, Double], count: Long): (Graph[Double, Double], VertexRDD[Int]) = {
  // Đếm từng giao dịch đi ra, kể cả các cạnh song song.
  val outDegrees = graph.outDegrees
  val degreeGraph = graph.outerJoinVertices(outDegrees) { (id, oldValue, degree) =>
    degree.getOrElse(0)
  }.persist(storageLevel)
  degreeGraph.vertices.count()

  // Dangling vertex là vertex không có cạnh đi ra.
  val danglingVertices = degreeGraph.vertices.filter { case (id, outDegree) =>
    outDegree == 0
  }.persist(storageLevel)
  danglingVertices.count()

  val weightedGraph = degreeGraph.mapTriplets(edge => 1.0 / edge.srcAttr, TripletFields.Src)
  val rankGraph = weightedGraph.mapVertices((id, outDegree) => 1.0 / count).persist(storageLevel)
  rankGraph.vertices.checkpoint()
  rankGraph.edges.checkpoint()
  rankGraph.vertices.count()
  rankGraph.edges.count()
  (rankGraph, danglingVertices)
}


### Một vòng lặp chỉ có ba công việc

Tính dangling mass, gửi rank contribution dọc cạnh và cập nhật tất cả vertex bằng `outerJoinVertices`.
Vertex không nhận message vẫn có teleport và dangling contribution.
Không dùng PageRank built-in vì benchmark cần cùng công thức và đúng số vòng với GraphFrames.

In [14]:
def pageRankStep(rankGraph: Graph[Double, Double], danglingVertices: VertexRDD[Int],
    count: Long, damping: Double): Graph[Double, Double] = {
  val danglingRanks = rankGraph.vertices.innerJoin(danglingVertices) { (id, rank, outDegree) =>
    rank
  }
  val danglingMass = danglingRanks.values.sum()

  val messages = rankGraph.aggregateMessages[Double](
    sendMsg = edge => {
      val contribution = edge.srcAttr * edge.attr
      edge.sendToDst(contribution)
    },
    mergeMsg = (left, right) => left + right,
    tripletFields = TripletFields.Src)

  val teleport = (1.0 - damping) / count
  val danglingShare = danglingMass / count
  rankGraph.outerJoinVertices(messages) { (id, oldRank, message) =>
    val incomingRank = message.getOrElse(0.0)
    teleport + damping * (incomingRank + danglingShare)
  }
}


### Chạy GraphX với các timer đã định nghĩa

Algorithm ở trên không chứa timer hoặc mã xuất CSV.
Đo đạc dưới đây chỉ gọi thuật toán và force action để tránh đo lazy plan thay vì tính toán thật.
Checkpoint mỗi vòng nằm trong timing và có thể ảnh hưởng chi phí tái sử dụng vertex index; đây là một giới hạn cần đọc cùng kết quả.

In [15]:
def runGraphX(saveEvidence: Boolean): Unit = {
  startPhase("build")
  val graph = buildGraphX(vertices, edges, partitionCount)
  require(graph.vertices.count() == vertexCount)
  require(graph.edges.count() == edgeCount)
  finishPhase()

  if (saveEvidence) {
    writeText(currentRun + "_rdd_dag.txt", graph.vertices.toDebugString)
  }
  val partitionLoads = graph.edges.mapPartitionsWithIndex { (partitionId, partitionEdges) =>
    Iterator((partitionId, partitionEdges.size.toLong))
  }.collect().toSeq
  saveLoad(partitionLoads)

  startPhase("aggregate")
  val incoming = incomingAmounts(graph).persist(storageLevel)
  incoming.count()
  incoming.values.sum()
  finishPhase()
  validateResult("incoming", incoming.collect().toMap, expectedIncoming)
  incoming.unpersist(blocking = true)

  startPhase("pr_setup")
  val (initialRankGraph, danglingVertices) = preparePageRank(graph, vertexCount)
  finishPhase()

  var rankGraph = initialRankGraph
  for (iteration <- 1 to iterations) {
    startPhase("pr_iteration", iteration)
    val nextGraph = pageRankStep(rankGraph, danglingVertices, vertexCount, damping)
    nextGraph.vertices.persist(storageLevel)
    nextGraph.vertices.checkpoint()
    val rankTotal = nextGraph.vertices.values.sum()
    require(math.abs(rankTotal - 1.0) < 1e-9, "PageRank total: " + rankTotal)
    rankGraph.unpersistVertices(blocking = true)
    rankGraph = nextGraph
    finishPhase()
  }
  validateResult("pagerank", rankGraph.vertices.collect().toMap, expectedRanks)
  if (saveEvidence) {
    writeText(currentRun + "_pr_rdd_dag.txt", rankGraph.vertices.toDebugString)
  }
  danglingVertices.unpersist(blocking = true)
}


## GraphFrames: cùng phép tính trên DataFrame

Dùng cùng USD amount và công thức PageRank như GraphX.
GraphFrames repartition theo src, AQE tắt; GraphX dùng EdgePartition2D.
Đây là hai cấu hình cố định, không đại diện mọi cách tối ưu của mỗi framework.
Checkpoint và materialization nằm trong timer.

In [16]:
def incomingFrame(graph: GraphFrame): DataFrame = {
  graph.aggregateMessages
    .sendToDst(AggregateMessages.edge("amount"))
    .agg(sum(AggregateMessages.msg).as("incoming"))
}


In [17]:
def prepareFrame(graph: GraphFrame, count: Long): (DataFrame, DataFrame) = {
  val degree = graph.edges.groupBy("src").count()
    .select(col("src").as("id"), col("count").as("out_degree"))

  val verticesWithDegree = graph.vertices.select("id").join(degree, Seq("id"), "left")
  val initialRanks = verticesWithDegree.na.fill(0L, Seq("out_degree"))
    .withColumn("rank", lit(1.0 / count))
  val state = initialRanks.checkpoint(eager = true).persist(storageLevel)
  state.count()

  val senderDegrees = degree.select(col("id").as("src"), col("out_degree"))
  val edgesWithDegree = graph.edges.join(senderDegrees, Seq("src"))
  val weighted = edgesWithDegree.select(col("src"), col("dst"),
    (lit(1.0) / col("out_degree")).as("weight"))
  val weightedEdges = weighted.repartition(partitionCount, col("src"))
    .checkpoint(eager = true).persist(storageLevel)
  weightedEdges.count()
  (state, weightedEdges)
}


In [18]:
def frameStep(state: DataFrame, weightedEdges: DataFrame, count: Long,
    damping: Double): (DataFrame, DataFrame) = {
  val danglingVertices = state.filter(col("out_degree") === 0L)
  val danglingTotal = danglingVertices.agg(coalesce(sum("rank"), lit(0.0)))
  val danglingMass = danglingTotal.first().getDouble(0)

  val graph = GraphFrame(state, weightedEdges)
  val contribution = AggregateMessages.src("rank") * AggregateMessages.edge("weight")
  val messages = graph.aggregateMessages.sendToDst(contribution)
    .agg(sum(AggregateMessages.msg).as("message"))

  val receivedMessages = state.select("id", "out_degree").join(messages, Seq("id"), "left")
  val incomingRank = coalesce(col("message"), lit(0.0))
  val teleport = (1.0 - damping) / count
  val danglingShare = danglingMass / count
  val nextRank = lit(teleport) + lit(damping) * (incomingRank + lit(danglingShare))
  val update = receivedMessages.select(col("id"), col("out_degree"), nextRank.as("rank"))

  // GraphFrames cache messages
  // Giải phóng sau khi next state đã được tính xong.
  (update, messages)
}


In [19]:
def runGraphFrames(saveEvidence: Boolean): Unit = {
  startPhase("build")
  val graphVertices = vertices.select(col("id"), lit(1.0).as("value"))
    .repartition(partitionCount, col("id")).persist(storageLevel)
  val graphEdges = edges.select("src", "dst", "amount")
    .repartition(partitionCount, col("src")).persist(storageLevel)
  val graph = GraphFrame(graphVertices, graphEdges)
  require(graph.vertices.count() == vertexCount)
  require(graph.edges.count() == edgeCount)
  finishPhase()

  val partitionLoads = graph.edges.groupBy(spark_partition_id().as("partition_id")).count()
  saveLoad(partitionLoads.as[(Int, Long)].collect().toSeq)

  startPhase("aggregate")
  val incoming = incomingFrame(graph).persist(storageLevel)
  incoming.count()
  incoming.agg(sum("incoming")).first()
  finishPhase()
  val incomingValues = incoming.select("id", "incoming").as[(Long, Double)].collect().toMap
  validateResult("incoming", incomingValues, expectedIncoming)
  if (saveEvidence) {
    writeText(currentRun + "_aggregate_plan.txt", incoming.queryExecution.executedPlan.toString)
  }
  incoming.unpersist(blocking = true)

  startPhase("pr_setup")
  val (initialState, weightedEdges) = prepareFrame(graph, vertexCount)
  finishPhase()

  var state = initialState
  for (iteration <- 1 to iterations) {
    startPhase("pr_iteration", iteration)
    val (update, messages) = frameStep(state, weightedEdges, vertexCount, damping)
    val nextState = update.checkpoint(eager = false).persist(storageLevel)
    val rankTotal = nextState.agg(sum("rank")).first().getDouble(0)
    require(math.abs(rankTotal - 1.0) < 1e-9, "PageRank total: " + rankTotal)
    messages.unpersist(blocking = true)
    state.unpersist(blocking = true)
    state = nextState
    finishPhase()
  }
  val rankValues = state.select("id", "rank").as[(Long, Double)].collect().toMap
  validateResult("pagerank", rankValues, expectedRanks)
  if (saveEvidence) {
    writeText(currentRun + "_pr_plan.txt", state.queryExecution.executedPlan.toString)
  }
  state.unpersist(blocking = true)
  weightedEdges.unpersist(blocking = true)
  graph.vertices.unpersist(blocking = true)
  graph.edges.unpersist(blocking = true)
}


## Reference độc lập

Kiểm tra incoming amount và PageRank tại từng vertex bằng Scala trên driver.
Dùng mảng primitive cho cạnh để giảm bộ nhớ; reference nằm ngoài timer.
Bộ nhớ driver vẫn tăng theo kích thước graph, cần heap phù hợp khi chạy toàn bộ HI-Small.

In [20]:
def reference(): Unit = {
  val ids = vertices.select("id").as[Long].collect().sorted
  // Mảng dùng vị trí 0..N-1
  // Mỗi ID được ánh xạ về vị trí trong mảng.
  val positions = ids.zipWithIndex.toMap
  require(edgeCount <= Int.MaxValue, "Too many edges for the driver reference")
  val sources = new Array[Int](edgeCount.toInt)
  val targets = new Array[Int](edgeCount.toInt)
  val incomingAmounts = Array.fill(ids.length)(0.0)
  val outDegrees = Array.fill(ids.length)(0L)

  val transactions = edges.select("src", "dst", "amount").as[(Long, Long, Double)].toLocalIterator()
  var edgeIndex = 0
  while (transactions.hasNext) {
    val (sender, receiver, amount) = transactions.next()
    val senderIndex = positions(sender)
    val receiverIndex = positions(receiver)
    sources(edgeIndex) = senderIndex
    targets(edgeIndex) = receiverIndex
    incomingAmounts(receiverIndex) += amount
    outDegrees(senderIndex) += 1
    edgeIndex += 1
  }
  expectedIncoming = ids.zip(incomingAmounts).toMap

  var ranks = Array.fill(ids.length)(1.0 / vertexCount)
  for (iteration <- 1 to iterations) {
    var danglingMass = 0.0
    for (index <- ids.indices) {
      if (outDegrees(index) == 0) {
        danglingMass += ranks(index)
      }
    }
    val baseRank = (1.0 - damping) / vertexCount + damping * danglingMass / vertexCount
    val nextRanks = Array.fill(ids.length)(baseRank)
    for (index <- sources.indices) {
      val senderIndex = sources(index)
      val receiverIndex = targets(index)
      val contribution = damping * ranks(senderIndex) / outDegrees(senderIndex)
      nextRanks(receiverIndex) += contribution
    }
    ranks = nextRanks
  }
  expectedRanks = ids.zip(ranks).toMap
}


In [21]:
reference()
println(s"Reference ready: ${expectedRanks.size} vertex ranks")

collect at cmd20.sc:2

hasNext at cmd20.sc:14

hasNext at cmd20.sc:14

Reference ready: 422734 vertex ranks


## Chạy benchmark

GraphX EdgePartition2D và GraphFrames src/AQE-OFF cùng nhận graph giao dịch đầy đủ.
Trộn thứ tự bằng seed cố định, dọn cache riêng của mỗi run và loại warmup khỏi thống kê.
Giữ cùng cache policy, tắt broadcast join và ghi event log để đối chiếu.

In [22]:
val frameworks = Seq("GraphX", "GraphFrames")
val random = new scala.util.Random(seed)
val schedule = new PrintWriter(new File(output, "schedule.csv"))
csv(schedule, Seq("run", "framework", "repeat", "warmup", "order"))
var completedRuns = 0


completedRuns: Int = 0

### Metadata

Lưu commit preprocessing được chọn, checksum của input, summary upstream và tham số benchmark.
Commit trong `upstream.json` ghi nguồn dự kiến; checksum định danh artifact thực tế, không tự chứng minh ai đã tạo artifact.
Đổi phiên bản preprocessing thì cập nhật lock và tạo bộ dữ liệu mới tương ứng.

In [23]:
def writeJson(file: File, fields: (String, Any)*): Unit = {
  val values = fields.toMap.asJava
  mapper.writerWithDefaultPrettyPrinter().writeValue(file, values)
}

// SHA-256 định danh chính xác file dữ liệu đã dùng.
def sha256(file: File): String = {
  val digest = MessageDigest.getInstance("SHA-256")
  val stream = Files.newInputStream(file.toPath)
  val buffer = new Array[Byte](1024 * 1024)
  try {
    var bytesRead = stream.read(buffer)
    while (bytesRead != -1) {
      digest.update(buffer, 0, bytesRead)
      bytesRead = stream.read(buffer)
    }
  } finally {
    stream.close()
  }
  val hexadecimal = new StringBuilder
  for (byte <- digest.digest()) {
    hexadecimal.append(String.format("%02x", Int.box(byte & 0xff)))
  }
  hexadecimal.toString()
}

def parquetFiles(path: File): Seq[File] = {
  val files = scala.collection.mutable.ArrayBuffer.empty[File]
  if (path.isFile) {
    files += path
  } else {
    require(path.isDirectory, "Missing Parquet path: " + path)
    for (child <- path.listFiles()) {
      if (child.isDirectory) {
        files ++= parquetFiles(child)
      } else if (child.getName.endsWith(".parquet")) {
        files += child
      }
    }
  }
  files.toSeq.sortBy(file => file.getPath)
}

val sourceFiles = scala.collection.mutable.ArrayBuffer.empty[File]
for (name <- Seq("vertices.parquet", "edges.parquet")) {
  val inputPath = new File(sourceDir, name)
  sourceFiles ++= parquetFiles(inputPath)
}
sourceFiles += summaryFile

val sourceHashes = scala.collection.mutable.Map.empty[String, String]
for (file <- sourceFiles) {
  val relativePath = sourceDir.toPath.relativize(file.toPath).toString
  sourceHashes(relativePath) = sha256(file)
}
writeJson(new File(output, "input_manifest.json"),
  "upstream_repository" -> upstream.get("repository").asText(),
  "upstream_commit" -> upstream.get("commit").asText(),
  "source_dir" -> sourceDir.getPath,
  "sha256" -> sourceHashes.toMap.asJava,
  "vertices" -> vertexCount, "edges" -> edgeCount, "edge_view" -> "transactions",
  "amount_column" -> "upstream amount (USD)", "sampling" -> "none",
  "vertices_path" -> new File(inputDir, "vertices").getPath,
  "edges_path" -> new File(inputDir, "edges").getPath,
  "id_encoding" -> "1-based index of sorted upstream IDs; source_id preserved")
Files.copy(summaryFile.toPath, new File(output, "preprocess_summary.json").toPath)
writeText("environment.txt", s"""spark=${spark.version}
scala=${scala.util.Properties.versionNumberString}
java=${System.getProperty("java.version")}
master=$master
vertices=$vertexCount
edges=$edgeCount
partitions=$partitionCount
iterations=$iterations
damping=$damping
seed=$seed
repeats=$repeats
warmups=$warmups
heap_max_bytes=${Runtime.getRuntime.maxMemory()}
input=upstream transaction edges
no sampling or pre-aggregation
execution=native Scala notebook
""")

In [24]:
try {
  // repeat âm là warmup. Các lượt đo chính có repeat = 0, 1, 2.
  for (repeat <- -warmups until repeats) {
    val frameworkOrder = random.shuffle(frameworks)
    for (framework <- frameworkOrder) {
      currentRun = framework + "_r" + repeat
      val isWarmup = repeat < 0
      val saveEvidence = repeat == 0
      csv(schedule, Seq(currentRun, framework, repeat, isWarmup, completedRuns))
      cleanup()
      try {
        if (framework == "GraphX") {
          runGraphX(saveEvidence)
        } else {
          runGraphFrames(saveEvidence)
        }
        completedRuns += 1
      } finally {
        cleanup()
      }
      println("Passed " + completedRuns + "/8: " + currentRun)
    }
  }
} finally {
  metrics.close()
  checks.close()
  loads.close()
  schedule.close()
  progress.close()
}


mapPartitions at VertexRDD.scala:356

map at GraphImpl.scala:100

map at cmd11.sc:3

mapPartitions at VertexRDD.scala:356

fold at VertexRDDImpl.scala:90

...

mapPartitions at GraphImpl.scala:207

sum at cmd15.sc:34

sum at cmd15.sc:34

collect at cmd15.sc:40

Passed 8/8: GraphX_r2


## 8. Đóng event log, công bố kết quả và chuyển sang Python

Chỉ cập nhật latest_run.json sau khi mọi cấu hình vượt qua correctness check.
Dừng Spark để event log hoàn tất, tránh Python đọc một lần chạy đang dở.
Notebook tiếp theo chỉ đọc output, không chạy lại Spark hoặc thay đổi workload.

In [25]:
require(completedRuns == frameworks.size * (warmups + repeats), "Benchmark chưa hoàvertexCount tất.")
vertices.unpersist(blocking = true)
edges.unpersist(blocking = true)
spark.stop()
writeText("SUCCESS", "All configurations matched independent incoming-sum and PageRank references.")
writeJson(new File(reportRoot, "latest_run.json"),
  "run_dir" -> root.toPath.relativize(output.toPath).toString,
  "workflow" -> "native-scala-notebook",
  "benchmark_notebook" -> "01_scala_graph_benchmark.ipynb")
println(s"Completed: ${frameworks.size * (warmups + repeats)} runs, ${2 * frameworks.size * (warmups + repeats)} correctness checks")
println(s"Output: ${output.getAbsolutePath}")
println("Next: open 02_python_visualize.ipynb and Run All.")

Completed: 8 runs, 16 correctness checks


Output: /home/tuanta/BigData/graphX/reports/runs/scala_1792579688023


Next: open 02_python_visualize.ipynb and Run All.


Runtime mô tả graph giao dịch và cấu hình của lần chạy này.
Trong local mode không có network giữa máy; durable checkpoint nằm trong timing.
Cache bytes không phải RSS hoặc peak heap.
Đọc notebook 02 để xem thời gian từng phase và tổng thời gian công việc đã đo.